# Track 1 — NaturalBoost V7: WavLM + IndicWav2Vec Complementary Fusion

**Goal:** improve the competition metric **Event F1 + Segment Dice** without discarding the strongest WavLM boundary model.

Paper-side evidence motivates a division of labour:

- **WavLM Base+**: strongest natural event F1 / boundary localization.
- **IndicWav2Vec Base**: strong segment overlap and synthetic-noise detection.

This notebook therefore keeps the proven **WavLM NaturalBoost V6 onset/offset heads** and trains a lightweight **Mel + IndicWav2Vec occupancy companion**. It then tunes only the **frame posterior fusion** while retaining WavLM onset/offset snapping.

The critical control is `alpha=1.0`, which exactly removes IndicWav2Vec from frame fusion. **Do not submit unless held-out Combined improves over this control.**

### Run order
1. Load the same V6 data/splits.
2. Reuse V6 WavLM checkpoints when present.
3. Reuse existing 768-D IndicWav2Vec cache and extract only missing clips.
4. Train the Indic occupancy companion.
5. Tune natural/synthetic fusion weights + decoder parameters.
6. Compare V7 against the WavLM-only control on untouched holdout halves.
7. Only if V7 wins, cache test Indic features and create the submission.


## 0 · Setup

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > T4 GPU")

GPU: Tesla T4


In [ ]:
!pip -q install librosa soundfile pyarrow pandas transformers scikit-learn scipy tqdm
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, glob, json, io, math, random, time, itertools, ast, zipfile, copy
import numpy as np, pandas as pd
import pyarrow.parquet as pqlib, soundfile as sf, librosa
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report
from sklearn.model_selection import train_test_split

DRIVE = "/content/drive/MyDrive/Vaani"
WORK  = "/content/drive/MyDrive/Datathon/work"
TEST_ROOT = "/content/drive/MyDrive/PHD/Datathon/"
CKPT = f"{WORK}/wavlm_plus_naturalboost_v6"
os.makedirs(CKPT, exist_ok=True)

SR,HOP,NFFT,N_MELS,FRAME = 16000,160,512,64,0.010
MAX_SEC = 10.0
MAX_FR = int(MAX_SEC/FRAME)

N_GOLD=None
N_SILVER=40000
SYN_FRAC=0.35
USE_MT=True
WAVLM="microsoft/wavlm-base-plus"
WAVLM_LAYERS=(6,9,12)          # hidden_states indexes for base-plus
TRAIN_SEEDS=(0,1)              # use (0,) for quick experiment
EPOCHS=22
BATCH_SIZE=8
LR=2e-4
SEED=1337

# Crop policy for NATURAL clips longer than MAX_SEC.
NAT_EVENT_CENTER_PROB=0.62
NAT_BACKGROUND_PROB=0.20
BOUNDARY_RADIUS_MS=80

# Loss weights
FRAME_GAMMA=1.7
DICE_W=0.35
ONSET_W=0.45
OFFSET_W=0.45
MT_MAX_W=1.2

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('WavLM:', WAVLM, '| layers:', WAVLM_LAYERS)
print('syn:', SYN_FRAC, '| silver:', N_SILVER, '| seeds:', TRAIN_SEEDS)

# ---------------- V7 additions ----------------
RETRAIN_WAVLM = False
INDIC_SEEDS = (0,1)
INDIC_EPOCHS = 14
INDIC_BATCH = 12
INDIC_LR = 2e-4
INDIC_DICE_W = 0.65

DUAL_CKPT = f"{WORK}/wavlm_indicw2v_dualrep_v7"
os.makedirs(DUAL_CKPT, exist_ok=True)

# Reuse the 768-D cache from the paper experiment whenever clip ids overlap.
IW2V_FEAT_DIR = "/content/drive/MyDrive/Datathon/globalsouth_indic_noise_probe/features/indicwav2vec_base"
os.makedirs(IW2V_FEAT_DIR, exist_ok=True)
IW2V_DIM = 768
IW2V_ENV = "/content/iw2v_env"
IW2V_FAIRSEQ = "/content/fairseq"
IW2V_CKPT = "/content/indicw2v_base_pretrained.pt"

# Fusion: alpha=1 => WavLM-only frame posterior; alpha=0 => Indic-only frame posterior.
ALPHAS_NAT = [1.00,0.95,0.90,0.85,0.80,0.75,0.70,0.60]
ALPHAS_SYN = [1.00,0.90,0.80,0.70,0.60,0.50,0.40,0.30]

print('V7 checkpoint dir:', DUAL_CKPT)
print('Indic feature dir:', IW2V_FEAT_DIR)


WavLM: microsoft/wavlm-base-plus | layers: (6, 9, 12)
syn: 0.35 | silver: 40000 | seeds: (0, 1)
V7 checkpoint dir: /content/drive/MyDrive/Datathon/work/wavlm_indicw2v_dualrep_v7
Indic feature dir: /content/drive/MyDrive/Datathon/globalsouth_indic_noise_probe/features/indicwav2vec_base


## 1 · Official-style local scorer

In [ ]:
"""Local re-implementation of Datathon@IndoML 2026 Track 1 metrics.

Event-based F1 : +/-20% of reference duration, min 50 ms, greedy closest-first matching.
Segment Dice   : 10 ms frames, binary masks, macro-averaged over clips.
"""
import numpy as np

FRAME = 0.010


def match_events(pred, ref):
    """Greedy closest-first matching. pred/ref are lists of (onset, offset).
    Returns (n_tp, pairs)."""
    cands = []
    for i, (po, pf) in enumerate(pred):
        for j, (ro, rf) in enumerate(ref):
            tol = max(0.20 * (rf - ro), 0.05)
            if abs(po - ro) <= tol and abs(pf - rf) <= tol:
                cands.append((abs(po - ro) + abs(pf - rf), i, j))
    cands.sort()
    up, ur, pairs = set(), set(), []
    for _, i, j in cands:
        if i in up or j in ur:
            continue
        up.add(i); ur.add(j); pairs.append((i, j))
    return len(pairs), pairs


def event_f1(preds, refs):
    """preds/refs: dict clip_id -> list of (onset, offset). Micro-averaged."""
    TP = FP = FN = 0
    for c in refs:
        p, r = preds.get(c, []), refs[c]
        tp, _ = match_events(p, r)
        TP += tp; FP += len(p) - tp; FN += len(r) - tp
    prec = TP / (TP + FP) if TP + FP else 0.0
    rec = TP / (TP + FN) if TP + FN else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return f1, prec, rec, TP, FP, FN


def _mask(events, n):
    m = np.zeros(n, dtype=bool)
    for on, off in events:
        a = max(0, int(round(on / FRAME)))
        b = min(n, int(round(off / FRAME)))
        if b > a:
            m[a:b] = True
    return m


def segment_dice(preds, refs, durs, empty_both=1.0):
    """Macro-averaged Dice. empty_both: score when BOTH pred and ref are empty."""
    out = []
    for c in refs:
        n = max(1, int(round(durs[c] / FRAME)))
        P = _mask(preds.get(c, []), n)
        G = _mask(refs[c], n)
        sp, sg = P.sum(), G.sum()
        if sp == 0 and sg == 0:
            out.append(empty_both)
        elif sp + sg == 0:
            out.append(empty_both)
        else:
            out.append(2.0 * np.logical_and(P, G).sum() / (sp + sg))
    return float(np.mean(out))


def score(preds, refs, durs, empty_both=1.0, verbose=True):
    f1, prec, rec, tp, fp, fn = event_f1(preds, refs)
    d = segment_dice(preds, refs, durs, empty_both)
    if verbose:
        print(f"  Event F1 : {f1:.4f}  (P {prec:.3f} R {rec:.3f} | TP {tp} FP {fp} FN {fn})")
        print(f"  Dice     : {d:.4f}")
        print(f"  Combined : {f1 + d:.4f}")
    return {"f1": f1, "precision": prec, "recall": rec, "dice": d, "combined": f1 + d}

In [ ]:
assert abs(segment_dice({'x':[(0,10)]},{'x':[(0,2)]},{'x':10.0})-2*0.2/1.2)<1e-6
print("scorer OK")

scorer OK


## 2 · Data and the same V6 supervised split

In [ ]:
FULL_TOL=0.10
def parse_events(v):
    if v is None: return []
    if isinstance(v,str):
        try: v=json.loads(v)
        except Exception:
            try: v=ast.literal_eval(v)
            except Exception: return []
    out=[]
    for a in (v if isinstance(v,(list,tuple,np.ndarray)) else []):
        if not isinstance(a,dict) or "start" not in a or "end" not in a: continue
        try: s,e=float(a["start"]),float(a["end"])
        except (TypeError,ValueError): continue
        if e>s: out.append((s,e,a.get("category"),a.get("tag")))
    return sorted(out)
def localized(ev,dur): return [x for x in ev if not (x[0]<=FULL_TOL and x[1]>=dur-FULL_TOL)]
def load_audio(au):
    raw=path=None
    if isinstance(au,dict): raw,path=au.get("bytes"),au.get("path")
    elif isinstance(au,(bytes,bytearray,memoryview)): raw=au
    elif isinstance(au,str): path=au
    if raw is not None and len(raw)>0:
        try: y,sr0=sf.read(io.BytesIO(bytes(raw)),dtype="float32")
        except Exception:
            try: y,sr0=librosa.load(io.BytesIO(bytes(raw)),sr=None,mono=True)
            except Exception: return None
    elif path and os.path.exists(path):
        try: y,sr0=librosa.load(path,sr=None,mono=True)
        except Exception: return None
    else: return None
    y=np.asarray(y,dtype=np.float32)
    if y.ndim>1: y=y.mean(1)
    if len(y)<int(0.2*SR): return None
    if sr0!=SR: y=librosa.resample(y,orig_sr=sr0,target_sr=SR)
    return y.astype(np.float32)

shards=sorted(glob.glob(f"{DRIVE}/parquet/part*/*.parquet")); assert shards
CACHE="/content/aud"; os.makedirs(CACHE,exist_ok=True)
gold,silver=[],[]; nfail=0; t0=time.time()
for si,sh in enumerate(shards):
    for r in pqlib.read_table(sh).to_pylist():
        q=r.get("annotationQuality")
        if   q=="verified_timestamps":                tgt,pre,cap=gold,"g",N_GOLD
        elif q=="unverified_timestamps" and N_SILVER: tgt,pre,cap=silver,"s",N_SILVER
        else: continue
        if cap and len(tgt)>=cap: continue
        y=load_audio(r["audio"])
        if y is None: nfail+=1; continue
        dur=len(y)/SR
        ev=[(s,min(e,dur),c,t) for s,e,c,t in parse_events(r.get("NoiseSubCategoryTimeStamp")) if s<dur]
        cid=f"{pre}{len(tgt):06d}"; np.save(f"{CACHE}/{cid}.npy",y)
        tgt.append(dict(cid=cid,path=f"{CACHE}/{cid}.npy",dur=dur,events=ev,
                        events_loc=localized(ev,dur),district=r.get("district","unk")))
    print(f"  shard {si+1}/{len(shards)}  gold {len(gold)}  silver {len(silver)}  {(time.time()-t0)/60:.1f}m",end="\r")
    if (not N_GOLD or len(gold)>=N_GOLD) and len(silver)>=N_SILVER: break
print(f"\ngold {len(gold)} ({sum(g['dur'] for g in gold)/3600:.1f}h) | silver {len(silver)} ({sum(g['dur'] for g in silver)/3600:.1f}h) | fail {nfail}")

  shard 115/182  gold 7081  silver 40000  8.0m
gold 7081 (14.2h) | silver 40000 (64.9h) | fail 2


In [ ]:
"""Synthetic augmentation: rebuild the test set's synthetic half from Gold.
Cut real noise events out of Gold clips, mix them into low-noise Gold clips at
known onsets -> strong labels for free, in exactly the distribution we're weak on."""
import numpy as np
SR, FRAME = 16000, 0.010

def build_noise_bank(gold, min_d=0.15, max_d=3.0, max_items=4000):
    bank=[]
    for g in gold:
        y=None
        for on,off,cat,tag in g.get("events_loc",g["events"]):
            d=off-on
            if not (min_d<=d<=max_d): continue
            if y is None: y=np.load(g["path"])
            a,b=int(on*SR),min(int(off*SR),len(y))
            if b-a<int(min_d*SR): continue
            seg=y[a:b].astype(np.float32)
            if np.sqrt((seg**2).mean()+1e-12)<1e-4: continue
            bank.append((seg,cat))
            if len(bank)>=max_items: return bank
    return bank

def pick_clean(gold, max_cov=0.02):
    out=[]
    for g in gold:
        ev=g.get("events_loc",g["events"])
        cov=sum(e-s for s,e,_,_ in ev)/max(g["dur"],1e-6)
        if cov<=max_cov: out.append(g)
    return out

def make_synthetic(clean_g, bank, rng, n_events=(1,3), snr_db=(0,20)):
    """Return (y, events) with events = list of (on,off,cat,tag)."""
    y=np.load(clean_g["path"]).astype(np.float32).copy()
    dur=len(y)/SR
    sp_rms=np.sqrt((y**2).mean()+1e-12)
    k=rng.integers(n_events[0],n_events[1]+1)
    ev=[]
    for _ in range(k):
        seg,cat=bank[rng.integers(len(bank))]
        if len(seg)>=len(y)-int(0.2*SR): continue
        a=int(rng.uniform(0,len(y)-len(seg)))
        snr=rng.uniform(*snr_db)
        n_rms=np.sqrt((seg**2).mean()+1e-12)
        gain=(sp_rms/(n_rms+1e-12))*(10**(-snr/20))
        y[a:a+len(seg)]+=gain*seg
        on,off=a/SR,(a+len(seg))/SR
        ev.append((on,min(off,dur),cat,"<syn>"))
    m=np.abs(y).max()
    if m>0.99: y=y/m*0.99
    # merge overlaps so labels stay consistent
    ev.sort(); out=[]
    for e in ev:
        if out and e[0]<=out[-1][1]: out[-1]=(out[-1][0],max(out[-1][1],e[1]),out[-1][2],out[-1][3])
        else: out.append(list(e) if False else e)
        if out and isinstance(out[-1],tuple) and len(out)>1 and out[-2][1]>=out[-1][0]:
            pass
    merged=[]
    for s,e_,c,t in sorted(ev):
        if merged and s<=merged[-1][1]: merged[-1]=(merged[-1][0],max(merged[-1][1],e_),merged[-1][2],merged[-1][3])
        else: merged.append((s,e_,c,t))
    return y, merged

In [ ]:
rng=np.random.default_rng(SEED)
BANK=build_noise_bank(gold); CLEAN=pick_clean(gold,max_cov=0.02)
N_SYN=int(SYN_FRAC/(1-SYN_FRAC)*len(gold))
SYNDIR="/content/syn"; os.makedirs(SYNDIR,exist_ok=True); syn=[]
for i in range(N_SYN):
    src=CLEAN[rng.integers(len(CLEAN))]
    y,ev=make_synthetic(src,BANK,rng,n_events=(1,3),snr_db=(0,20))
    if not ev: continue
    cid=f"y{i:06d}"; p=f"{SYNDIR}/{cid}.npy"; np.save(p,y.astype(np.float32))
    syn.append(dict(cid=cid,path=p,dur=len(y)/SR,events=ev,events_loc=ev,district=src["district"]))
print(f"{len(syn)} synthetic = {len(syn)/(len(gold)+len(syn)):.1%} of supervised (target {SYN_FRAC:.0%})")

3811 synthetic = 35.0% of supervised (target 35%)


### Speaker-disjoint split (same V6 policy)

In [ ]:
"""Speaker-disjoint dev split, matching the test set's actual generalisation condition.

Test clip_id structure (verified against real filenames):
  IISc_VaaniProject_<vendor>_<state>_<district>_<field5>_<field6>_<image>_...
Vendor K: field5 = district + long numeric session id  -> field5 IS the speaker/session key.
Vendor M/S: field5 = numeric participant id (no district prefix) -> field5 IS the speaker key.
So: field5, taken as-is, is the speaker identifier across all three vendors.

Gold's HF schema doesn't expose this field directly (no per-row speaker id column), so we fall
back to DISTRICT when no speaker id is parseable. That fallback is STRICTER than speaker-disjoint
(fewer, larger groups), so results on it are a lower bound / safe estimate, never an over-estimate
of generalisation.
"""
import re

def speaker_key(name_or_id, district=None):
    """Best-effort speaker/session key from a clip_id or imageFileName-like string.
    district: pass the row's district when available - M/S-vendor participant ids are bare
    numbers with no district prefix, so two speakers in different districts could in principle
    share a numeric id. Prefixing with district removes that (unlikely but free) risk."""
    if not name_or_id:
        return None
    parts = [p for p in re.split(r'[_/]', name_or_id) if p]
    for p in parts:
        if re.fullmatch(r'[A-Za-z]+\d{6,}', p):        # K-style: District123456789 (already district-scoped)
            return p
        if re.fullmatch(r'\d{6,}', p):                 # M/S-style: bare numeric participant id
            return f"{district}::{p}" if district else p
    return None

def speaker_disjoint_split(items, val_frac=0.2, seed=1337,
                           key_field="cid", fallback_field="district"):
    """items: list of dicts. Splits on speaker_key(item[key_field]); falls back to
    item[fallback_field] (district) when no key is parseable - a STRICTER split, never looser."""
    import random
    groups = {}
    n_fallback = 0
    for it in items:
        k = speaker_key(it.get(key_field, ""), district=it.get(fallback_field))
        if k is None:
            k = f"district::{it.get(fallback_field, 'unk')}"
            n_fallback += 1
        groups.setdefault(k, []).append(it)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    n_val = max(1, int(val_frac * len(keys)))
    val_keys = set(keys[:n_val])
    train = [it for k in keys if k not in val_keys for it in groups[k]]
    val   = [it for k in val_keys for it in groups[k]]
    return train, val, groups, n_fallback

In [ ]:
# speaker-disjoint (with district fallback), applied consistently across gold/syn/silver
gtr,gva,g_groups,g_fb = speaker_disjoint_split(gold, key_field="cid", fallback_field="district")
str_,sva,_,_          = speaker_disjoint_split(syn,  key_field="cid", fallback_field="district")
siltr,_,_,_           = speaker_disjoint_split(silver, key_field="cid", fallback_field="district")
print(f"gold: {len(g_groups)} speaker groups, {g_fb} used district fallback ({g_fb/len(gold):.1%})")
sup_train=gtr+str_
print(f"supervised {len(sup_train)} ({len(gtr)} gold + {len(str_)} syn) | unlabelled {len(siltr)}")
print(f"val {len(gva)} natural | {len(sva)} synthetic")

gold: 7081 speaker groups, 0 used district fallback (0.0%)
supervised 8714 (5665 gold + 3049 syn) | unlabelled 32000
val 1416 natural | 762 synthetic


## 3 · WavLM V6 boundary model

WavLM remains responsible for **onset/offset boundaries**. If the V6 checkpoints already exist in Drive, this notebook loads them instead of retraining.

In [ ]:
# ============================================================
# FIXED FrozenWavLMPlus
# Keeps frozen WavLM permanently in eval mode
# ============================================================

class FrozenWavLMPlus(nn.Module):

    def __init__(
        self,
        name=WAVLM,
        layer_ids=WAVLM_LAYERS
    ):
        super().__init__()

        from transformers import WavLMModel

        self.m = WavLMModel.from_pretrained(name)

        # Freeze backbone
        for p in self.m.parameters():
            p.requires_grad_(False)

        self.m.eval()

        self.layer_ids = tuple(layer_ids)

        self.layer_logits = nn.Parameter(
            torch.zeros(len(self.layer_ids))
        )

        self.dim = self.m.config.hidden_size

        print(
            "WavLM hidden layers:",
            self.m.config.num_hidden_layers,
            "| selected:",
            self.layer_ids,
            "| dim:",
            self.dim
        )

    # --------------------------------------------------------
    # CRITICAL FIX:
    # parent_model.train() must NOT switch frozen WavLM
    # back into training / LayerDrop mode.
    # --------------------------------------------------------
    def train(self, mode=True):
        super().train(mode)

        # force backbone to remain eval
        self.m.eval()

        return self

    def forward(
        self,
        wav,
        T_target
    ):

        # belt + suspenders:
        # guarantee no LayerDrop even if parent changed state
        self.m.eval()

        with torch.no_grad():

            out = self.m(
                wav,
                output_hidden_states=True,
                return_dict=True
            )

            hs_all = out.hidden_states

            # diagnostic / safety
            n_hs = len(hs_all)

            if max(self.layer_ids) >= n_hs:
                raise RuntimeError(
                    f"WavLM returned only {n_hs} hidden states "
                    f"but requested {self.layer_ids}. "
                    f"training={self.m.training}"
                )

            hs = [
                hs_all[i].float()
                for i in self.layer_ids
            ]

        # Learned layer mixture remains trainable
        w = torch.softmax(
            self.layer_logits,
            dim=0
        )

        h = sum(
            wi * hi
            for wi, hi in zip(w, hs)
        )

        # WavLM ~20 ms -> detector 10 ms grid
        h = F.interpolate(
            h.transpose(1, 2),
            size=T_target,
            mode="linear",
            align_corners=False
        )

        return h.transpose(1, 2)


print("✓ Fixed FrozenWavLMPlus defined")


class WavLMPlusBoundaryNet(nn.Module):
    def __init__(self, n_mels=N_MELS, ch=(32,64,128,128), rnn=224, wavlm_proj=160):
        super().__init__()
        blk=[]; cin=1
        for c in ch:
            blk += [nn.Conv2d(cin,c,3,padding=1), nn.BatchNorm2d(c), nn.SiLU(),
                    nn.AvgPool2d((2,1)), nn.Dropout2d(0.10)]
            cin=c
        self.cnn=nn.Sequential(*blk)
        self.mel_feat=cin*(n_mels//2**len(ch))
        self.wavlm=FrozenWavLMPlus()
        self.wav_proj=nn.Sequential(
            nn.Linear(self.wavlm.dim,wavlm_proj), nn.LayerNorm(wavlm_proj), nn.SiLU(), nn.Dropout(0.10)
        )
        feat=self.mel_feat+wavlm_proj
        self.rnn=nn.GRU(feat,rnn,2,batch_first=True,bidirectional=True,dropout=0.20)
        self.norm=nn.LayerNorm(2*rnn)
        self.frame_head=nn.Linear(2*rnn,1)
        self.onset_head=nn.Linear(2*rnn,1)
        self.offset_head=nn.Linear(2*rnn,1)

    def forward(self,x,wav):
        h=self.cnn(x)
        B,C,Fq,T=h.shape
        h=h.permute(0,3,1,2).reshape(B,T,C*Fq)
        w=self.wav_proj(self.wavlm(wav,T))
        h=torch.cat([h,w],dim=-1)
        h,_=self.rnn(h)
        h=self.norm(h)
        return {
            'frame': self.frame_head(h).squeeze(-1),
            'onset': self.onset_head(h).squeeze(-1),
            'offset': self.offset_head(h).squeeze(-1),
        }

_tmp=WavLMPlusBoundaryNet()
trainable=sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
total=sum(p.numel() for p in _tmp.parameters())
print(f'total {total/1e6:.1f}M | trainable {trainable/1e6:.2f}M')
del _tmp

✓ Fixed FrozenWavLMPlus defined


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

WavLM hidden layers: 12 | selected: (6, 9, 12) | dim: 768
total 96.9M | trainable 2.48M


In [ ]:
# ============================================================
# SANITY CHECK
# ============================================================

_test = FrozenWavLMPlus().cuda()

print(
    "Before outer train():",
    "wrapper training =", _test.training,
    "| WavLM training =", _test.m.training
)

_test.train()

print(
    "After outer train():",
    "wrapper training =", _test.training,
    "| WavLM training =", _test.m.training
)

# random 2-second waveform
wav = torch.randn(
    2,
    SR * 2,
    device="cuda"
)

with torch.no_grad():
    z = _test(
        wav,
        T_target=200
    )

print("output:", z.shape)

assert _test.m.training is False
assert z.shape == (2, 200, 768)

del _test, wav, z
torch.cuda.empty_cache()

print("✓ WavLM fixed: LayerDrop disabled")

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

WavLM hidden layers: 12 | selected: (6, 9, 12) | dim: 768
Before outer train(): wrapper training = True | WavLM training = False
After outer train(): wrapper training = True | WavLM training = False
output: torch.Size([2, 200, 768])
✓ WavLM fixed: LayerDrop disabled


## 4 · Shared mel/crop utilities and losses

In [ ]:
mel_fb=librosa.filters.mel(sr=SR,n_fft=NFFT,n_mels=N_MELS,fmax=8000)

def logmel(y):
    S=np.abs(librosa.stft(y,n_fft=NFFT,hop_length=HOP,center=True))**2
    return np.log(mel_fb@S+1e-8).astype(np.float32)

def spec_augment(x,n_t=2,n_f=2,max_t=35,max_f=8):
    x=x.copy(); Fq,T=x.shape; mn=x.mean()
    for _ in range(n_t):
        w=random.randint(0,max_t)
        if w>0 and T>w:
            s=random.randint(0,T-w); x[:,s:s+w]=mn
    for _ in range(n_f):
        w=random.randint(0,max_f)
        if w>0 and Fq>w:
            s=random.randint(0,Fq-w); x[s:s+w,:]=mn
    return x

def triangular_boundary_target(T, centers, radius_frames):
    z=np.zeros(T,np.float32)
    for c in centers:
        c=int(c)
        a=max(0,c-radius_frames); b=min(T,c+radius_frames+1)
        idx=np.arange(a,b)
        vals=1.0-np.abs(idx-c)/max(radius_frames,1)
        z[a:b]=np.maximum(z[a:b],np.clip(vals,0,1))
    return z

def choose_natural_crop(frame_target, events, T):
    if T<=MAX_FR: return 0
    r=random.random()
    if events and r < NAT_EVENT_CENTER_PROB:
        on,off,_,_=random.choice(events)
        center=int(0.5*(on+off)/FRAME)
        jitter=random.randint(-int(0.8/FRAME),int(0.8/FRAME))
        return int(np.clip(center+jitter-MAX_FR//2,0,T-MAX_FR))
    if r < NAT_EVENT_CENTER_PROB + NAT_BACKGROUND_PROB:
        # pick a background-rich crop; try several candidates and take lowest positive occupancy
        best=(1e9,0)
        for _ in range(12):
            st=random.randint(0,T-MAX_FR)
            occ=float(frame_target[st:st+MAX_FR].mean())
            if occ<best[0]: best=(occ,st)
        return best[1]
    return random.randint(0,T-MAX_FR)

class BoundarySup(Dataset):
    def __init__(self,items,train=True):
        self.items=items; self.train=train
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        g=self.items[i]
        y=np.load(g['path']).astype(np.float32)
        x=logmel(y); T=x.shape[1]
        frame=np.zeros(T,np.float32)
        ons=[]; offs=[]
        for on,off,_,_ in g['events_loc']:
            a=max(0,int(round(on/FRAME))); b=min(T,int(round(off/FRAME)))
            if b>a:
                frame[a:b]=1.0; ons.append(a); offs.append(max(a,b-1))
        rad=max(2,int(round((BOUNDARY_RADIUS_MS/1000)/FRAME)))
        onset=triangular_boundary_target(T,ons,rad)
        offset=triangular_boundary_target(T,offs,rad)

        if T>MAX_FR:
            is_syn=str(g['cid']).startswith('y')
            if self.train and not is_syn:
                st=choose_natural_crop(frame,g['events_loc'],T)
            else:
                st=random.randint(0,T-MAX_FR) if self.train else 0
            x=x[:,st:st+MAX_FR]; frame=frame[st:st+MAX_FR]
            onset=onset[st:st+MAX_FR]; offset=offset[st:st+MAX_FR]
            y=y[st*HOP:(st+MAX_FR)*HOP]

        xn=(x-x.mean())/(x.std()+1e-5)
        if self.train: xn=spec_augment(xn)
        need=x.shape[1]*HOP
        y=np.pad(y,(0,max(0,need-len(y))))[:need]
        return (torch.from_numpy(xn),torch.from_numpy(frame),torch.from_numpy(onset),
                torch.from_numpy(offset),torch.from_numpy(y.astype(np.float32)))

class BoundaryUnsup(Dataset):
    def __init__(self,items): self.items=items
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        g=self.items[i]; y=np.load(g['path']).astype(np.float32); x=logmel(y); T=x.shape[1]
        if T>MAX_FR:
            st=random.randint(0,T-MAX_FR); x=x[:,st:st+MAX_FR]; y=y[st*HOP:(st+MAX_FR)*HOP]
        xn=(x-x.mean())/(x.std()+1e-5)
        need=x.shape[1]*HOP; y=np.pad(y,(0,max(0,need-len(y))))[:need]
        return torch.from_numpy(xn.copy()),torch.from_numpy(spec_augment(xn)),torch.from_numpy(y)

def collate_sup(b):
    T=max(z[0].shape[1] for z in b); W=T*HOP; B=len(b)
    X=torch.zeros(B,1,N_MELS,T); Fm=torch.zeros(B,T); On=torch.zeros(B,T); Off=torch.zeros(B,T)
    M=torch.zeros(B,T); A=torch.zeros(B,W)
    for i,(x,f,o,q,w) in enumerate(b):
        n=x.shape[1]; X[i,0,:,:n]=x; Fm[i,:n]=f; On[i,:n]=o; Off[i,:n]=q; M[i,:n]=1
        A[i,:min(W,len(w))]=w[:W]
    return X,Fm,On,Off,M,A

def collate_unsup(b):
    T=max(z[0].shape[1] for z in b); W=T*HOP; B=len(b)
    P=torch.zeros(B,1,N_MELS,T); Q=torch.zeros(B,1,N_MELS,T); M=torch.zeros(B,T); A=torch.zeros(B,W)
    for i,(p,q,w) in enumerate(b):
        n=p.shape[1]; P[i,0,:,:n]=p; Q[i,0,:,:n]=q; M[i,:n]=1; A[i,:min(W,len(w))]=w[:W]
    return P,Q,M,A

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

In [ ]:
from collections import Counter

cat_counts=Counter()
for g in gtr:
    cats=set((c or 'unknown') for _,_,c,_ in g.get('events_loc',[]))
    for c in cats: cat_counts[c]+=1

def sample_weight(g):
    if str(g['cid']).startswith('y'):
        return 0.70  # preserve synthetic strength, but stop it dominating easy batches
    cats=set((c or 'unknown') for _,_,c,_ in g.get('events_loc',[]))
    if not cats: return 0.75
    # rare natural categories get more repeats; capped to avoid instability
    vals=[math.sqrt(max(cat_counts.values())/max(1,cat_counts[c])) for c in cats]
    return float(np.clip(max(vals),1.0,3.0))

SUP_WEIGHTS=np.asarray([sample_weight(g) for g in sup_train],np.float64)
print('sampler weight percentiles:',np.percentile(SUP_WEIGHTS,[0,25,50,75,95,100]))

model.safetensors: downloading bytes:           |  0.00B            

sampler weight percentiles: [0.7        0.7        1.         1.55942312 2.06031501 3.        ]


In [ ]:
def masked_focal_bce(logits,target,mask,gamma=FRAME_GAMMA,pos_boost=1.6):
    bce=F.binary_cross_entropy_with_logits(logits,target,reduction='none')
    p=torch.sigmoid(logits)
    pt=p*target+(1-p)*(1-target)
    focal=(1-pt).pow(gamma)
    w=1+(pos_boost-1)*target
    return (bce*focal*w*mask).sum()/mask.sum().clamp(min=1)

def soft_dice_loss(logits,target,mask):
    p=torch.sigmoid(logits)*mask; t=target*mask
    inter=(p*t).sum(dim=1)
    den=p.sum(dim=1)+t.sum(dim=1)
    return (1-(2*inter+1)/(den+1)).mean()

def boundary_bce(logits,target,mask,pos_boost=5.0):
    b=F.binary_cross_entropy_with_logits(logits,target,reduction='none')
    w=1+(pos_boost-1)*target
    return (b*w*mask).sum()/mask.sum().clamp(min=1)

class EMATeacher:
    def __init__(self,model,decay=0.999):
        self.model=copy.deepcopy(model).eval()
        for p in self.model.parameters(): p.requires_grad_(False)
        self.decay=decay
    @torch.no_grad()
    def update(self,student,step):
        d=min(self.decay,(step+1)/(step+10))
        for tp,sp in zip(self.model.parameters(),student.parameters()):
            tp.mul_(d).add_(sp.detach(),alpha=1-d)
        for tb,sb in zip(self.model.buffers(),student.buffers()): tb.copy_(sb)

def ramp_up(step,total,max_w=MT_MAX_W):
    p=np.clip(step/max(total,1),0,1)
    return float(max_w*np.exp(-5*(1-p)**2))

def consistency_loss(a,b,mask):
    pa=torch.sigmoid(a); pb=torch.sigmoid(b)
    return (((pa-pb)**2)*mask).sum()/mask.sum().clamp(min=1)

In [ ]:
# ============================================================
# GPU / PYTORCH DIAGNOSTIC
# ============================================================

import torch, subprocess

print("PyTorch version :", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("CUDA available  :", torch.cuda.is_available())

print("\nNVIDIA GPU:")
!nvidia-smi

PyTorch version : 2.11.0+cu128
Torch CUDA build: 12.8
CUDA available  : True

NVIDIA GPU:
Thu Sep  3 15:57:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |       

## 5 · Load/retrain the WavLM V6 ensemble

In [19]:
def fit_v6(tag,seed=0):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    model=WavLMPlusBoundaryNet().cuda()
    ckpt_path=f'{CKPT}/{tag}.pt'
    if (not RETRAIN_WAVLM) and os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path,map_location='cuda'))
        model.eval()
        print(f'{tag}: loaded existing V6 checkpoint')
        return model
    sampler=WeightedRandomSampler(SUP_WEIGHTS,len(sup_train),replacement=True)
    sl=DataLoader(BoundarySup(sup_train,True),batch_size=BATCH_SIZE,sampler=sampler,
                  collate_fn=collate_sup,num_workers=2,drop_last=True,persistent_workers=True)
    ul=(DataLoader(BoundaryUnsup(siltr),batch_size=BATCH_SIZE,shuffle=True,
                   collate_fn=collate_unsup,num_workers=2,drop_last=True,persistent_workers=True) if USE_MT else None)
    vl=DataLoader(BoundarySup(gva,False),batch_size=BATCH_SIZE,shuffle=False,
                  collate_fn=collate_sup,num_workers=2)

    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR,weight_decay=1e-2)
    steps=EPOCHS*len(sl)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=LR,total_steps=steps,pct_start=0.12)
    scaler=torch.amp.GradScaler('cuda')
    teacher=EMATeacher(model) if ul else None
    if teacher: teacher.model.cuda()
    best=1e9; step=0

    for ep in range(EPOCHS):
        model.train(); uit=iter(ul) if ul else None; tot=n=0
        for X,Y,On,Off,M,A in sl:
            X,Y,On,Off,M,A=[z.cuda(non_blocking=True) for z in (X,Y,On,Off,M,A)]
            with torch.amp.autocast('cuda'):
                o=model(X,A)
                lf=masked_focal_bce(o['frame'],Y,M)
                ld=soft_dice_loss(o['frame'],Y,M)
                lon=boundary_bce(o['onset'],On,M)
                lof=boundary_bce(o['offset'],Off,M)
                loss=lf+DICE_W*ld+ONSET_W*lon+OFFSET_W*lof
                if ul:
                    try: P,Q,UM,UA=next(uit)
                    except StopIteration: uit=iter(ul); P,Q,UM,UA=next(uit)
                    P,Q,UM,UA=[z.cuda(non_blocking=True) for z in (P,Q,UM,UA)]
                    with torch.no_grad(): to=teacher.model(P,UA)['frame']
                    so=model(Q,UA)['frame']
                    loss=loss+ramp_up(step,steps*0.45)*consistency_loss(so,to,UM)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],5.0)
            scaler.step(opt); scaler.update(); sch.step()
            if teacher: teacher.update(model,step)
            step+=1; tot+=loss.item()*len(X); n+=len(X)

        # validation objective emphasises frame/boundary quality but checkpoint selection is stable and cheap
        model.eval(); vt=vn=0
        with torch.no_grad():
            for X,Y,On,Off,M,A in vl:
                X,Y,On,Off,M,A=[z.cuda() for z in (X,Y,On,Off,M,A)]
                with torch.amp.autocast('cuda'):
                    o=model(X,A)
                    v=(masked_focal_bce(o['frame'],Y,M)+DICE_W*soft_dice_loss(o['frame'],Y,M)+
                       ONSET_W*boundary_bce(o['onset'],On,M)+OFFSET_W*boundary_bce(o['offset'],Off,M))
                vt+=v.item()*len(X); vn+=len(X)
        va=vt/max(vn,1); flag=''
        if va<best:
            best=va; torch.save(model.state_dict(),ckpt_path); flag=' *'
        lw=torch.softmax(model.wavlm.layer_logits.detach().cpu(),0).numpy()
        print(f'{tag} ep {ep+1:02d} train={tot/max(n,1):.4f} val={va:.4f}{flag} layer_w={np.round(lw,3)}')

    model.load_state_dict(torch.load(ckpt_path,map_location='cuda'))
    model.eval(); return model

models=[]
for s in TRAIN_SEEDS:
    models.append(fit_v6(f'v6_s{s}',seed=s))
print('trained models:',len(models))

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

WavLM hidden layers: 12 | selected: (6, 9, 12) | dim: 768
v6_s0 ep 01 train=0.6871 val=0.6146 * layer_w=[0.338 0.331 0.33 ]
v6_s0 ep 02 train=0.5969 val=0.5824 * layer_w=[0.362 0.315 0.323]
v6_s0 ep 03 train=0.5578 val=0.5566 * layer_w=[0.386 0.293 0.32 ]
v6_s0 ep 04 train=0.5352 val=0.5338 * layer_w=[0.405 0.279 0.316]
v6_s0 ep 05 train=0.5154 val=0.5398 layer_w=[0.421 0.266 0.313]
v6_s0 ep 06 train=0.4923 val=0.5414 layer_w=[0.434 0.257 0.309]
v6_s0 ep 07 train=0.4742 val=0.5575 layer_w=[0.445 0.249 0.306]
v6_s0 ep 08 train=0.4462 val=0.5714 layer_w=[0.453 0.244 0.303]
v6_s0 ep 09 train=0.4244 val=0.5911 layer_w=[0.46 0.24 0.3 ]
v6_s0 ep 10 train=0.4000 val=0.5974 layer_w=[0.465 0.238 0.298]
v6_s0 ep 11 train=0.3848 val=0.6279 layer_w=[0.468 0.236 0.296]
v6_s0 ep 12 train=0.3594 val=0.6401 layer_w=[0.472 0.234 0.294]
v6_s0 ep 13 train=0.3436 val=0.7004 layer_w=[0.474 0.233 0.293]
v6_s0 ep 14 train=0.3252 val=0.7217 layer_w=[0.476 0.232 0.292]
v6_s0 ep 15 train=0.3136 val=0.7353 layer

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

WavLM hidden layers: 12 | selected: (6, 9, 12) | dim: 768
v6_s1 ep 01 train=0.6901 val=0.6086 * layer_w=[0.337 0.332 0.33 ]
v6_s1 ep 02 train=0.5982 val=0.5678 * layer_w=[0.358 0.318 0.324]
v6_s1 ep 03 train=0.5648 val=0.5556 * layer_w=[0.379 0.301 0.32 ]
v6_s1 ep 04 train=0.5352 val=0.5496 * layer_w=[0.397 0.285 0.318]
v6_s1 ep 05 train=0.5168 val=0.5356 * layer_w=[0.412 0.272 0.316]
v6_s1 ep 06 train=0.4939 val=0.5482 layer_w=[0.424 0.262 0.314]


KeyboardInterrupt: 

## 6 · IndicWav2Vec 768-D contextual feature cache

The AI4Bharat fairseq checkpoint needs an older Python stack. We isolate it in a Python 3.10 `uv` environment and write only frozen `.npy` features back to Drive.

The extractor explicitly requests the **contextual 768-D transformer output**. Existing 512-D convolutional-cache files are treated as invalid and repaired. For the current V7 training, only `sup_train + gva + sva` are required; the 40k silver clips are **not** extracted.

In [20]:
import subprocess, pathlib, json, textwrap, os, sys, numpy as np
from pathlib import Path

# Generic extractor script: supports .npy training audio and .wav test audio.
EXTRACT_SCRIPT = "/content/extract_indicwav2vec_v7.py"
script = r'''
import os, sys, json
import numpy as np
import torch
import torch.nn.functional as F
import soundfile as sf
from scipy.signal import resample_poly
from tqdm import tqdm

sys.path.insert(0, "/content/fairseq")
from fairseq import checkpoint_utils

CKPT = "/content/indicw2v_base_pretrained.pt"
OUTDIR = "/content/drive/MyDrive/Datathon/globalsouth_indic_noise_probe/features/indicwav2vec_base"
EXPECTED_DIM = 768
SR = 16000

manifest = sys.argv[1]
os.makedirs(OUTDIR, exist_ok=True)
models,cfg,task = checkpoint_utils.load_model_ensemble_and_task([CKPT])
device = "cuda" if torch.cuda.is_available() else "cpu"
model = models[0].to(device).eval()
for p in model.parameters(): p.requires_grad_(False)
try: NORMALIZE = bool(cfg.task.normalize)
except Exception: NORMALIZE = False
print("IndicWav2Vec device:",device,"normalize:",NORMALIZE)


def load_audio(path):
    if path.lower().endswith('.npy'):
        y=np.load(path).astype(np.float32); sr=SR
    else:
        y,sr=sf.read(path,dtype='float32')
        y=np.asarray(y,np.float32)
        if y.ndim>1: y=y.mean(axis=1)
    if sr!=SR:
        from math import gcd
        g=gcd(int(sr),SR)
        y=resample_poly(y,SR//g,int(sr)//g).astype(np.float32)
    return y.astype(np.float32)

@torch.inference_mode()
def encode(y):
    wav=torch.from_numpy(np.asarray(y,np.float32)).unsqueeze(0).to(device)
    if NORMALIZE: wav=F.layer_norm(wav,wav.shape[1:])
    pm=torch.zeros(wav.shape,dtype=torch.bool,device=device)
    out=model(source=wav,padding_mask=pm,mask=False,features_only=True)
    if isinstance(out,dict): h=out['x']
    elif isinstance(out,(tuple,list)): h=out[0]
    else: h=out
    h=h[0]
    if h.ndim!=2: raise RuntimeError(f"bad feature rank {tuple(h.shape)}")
    if h.shape[1]==EXPECTED_DIM: pass
    elif h.shape[0]==EXPECTED_DIM: h=h.T
    else: raise RuntimeError(f"Expected contextual 768-D features, got {tuple(h.shape)}")
    return h.float().cpu().numpy().astype(np.float16)

items=[json.loads(x) for x in open(manifest,encoding='utf-8') if x.strip()]
for g in tqdm(items,desc='IndicWav2Vec cache'):
    fp=os.path.join(OUTDIR,g['cid']+'.npy')
    good=False
    if os.path.exists(fp):
        try:
            z=np.load(fp,mmap_mode='r'); good=(z.ndim==2 and z.shape[1]==EXPECTED_DIM)
        except Exception: good=False
    if good: continue
    y=load_audio(g['path'])
    z=encode(y)
    np.save(fp,z)
print('done',len(items))
'''
open(EXTRACT_SCRIPT,'w').write(script)


def ensure_iw2v_runtime():
    if not os.path.exists(IW2V_ENV + "/bin/python"):
        subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
        subprocess.run(['uv','venv',IW2V_ENV,'--python','3.10'],check=True)
        subprocess.run(['uv','pip','install','--python',IW2V_ENV+'/bin/python',
                        'torch==2.1.2','torchaudio==2.1.2','numpy==1.26.4','scipy','soundfile',
                        'omegaconf==2.0.6','hydra-core==1.0.7','bitarray','cython','tqdm'],check=True)
    if not os.path.exists(IW2V_FAIRSEQ):
        subprocess.run(['git','clone','-q','https://github.com/AI4Bharat/fairseq.git',IW2V_FAIRSEQ],check=True)
    # Editable install is idempotent enough for this isolated env.
    subprocess.run(['uv','pip','install','--python',IW2V_ENV+'/bin/python','-e',IW2V_FAIRSEQ],check=True)
    if not os.path.exists(IW2V_CKPT):
        subprocess.run(['wget','-q','https://objectstore.e2enetworks.net/indic-superb/aaai_ckpts/pretrained_models/indicw2v_base_pretrained.pt','-O',IW2V_CKPT],check=True)


def bad_or_missing(items):
    bad=[]
    for g in items:
        fp=f"{IW2V_FEAT_DIR}/{g['cid']}.npy"
        ok=False
        if os.path.exists(fp):
            try:
                z=np.load(fp,mmap_mode='r'); ok=(z.ndim==2 and z.shape[1]==IW2V_DIM)
            except Exception: ok=False
        if not ok: bad.append(g)
    return bad


def run_iw2v_cache(items, manifest='/content/iw2v_v7_manifest.jsonl'):
    missing=bad_or_missing(items)
    print(f'Indic cache required {len(items)} | already good {len(items)-len(missing)} | extract/repair {len(missing)}')
    if not missing: return
    with open(manifest,'w',encoding='utf-8') as f:
        for g in missing:
            f.write(json.dumps({'cid':g['cid'],'path':g['path']})+'\\n')
    ensure_iw2v_runtime()
    subprocess.run([IW2V_ENV+'/bin/python',EXTRACT_SCRIPT,manifest],check=True)
    left=bad_or_missing(items)
    assert not left, f'{len(left)} invalid Indic features remain'

REQUIRED_IW2V = sup_train + gva + sva
run_iw2v_cache(REQUIRED_IW2V)
print('✓ supervised/calibration Indic cache ready')

Indic cache required 10892 | already good 3600 | extract/repair 7292


CalledProcessError: Command '['/content/iw2v_env/bin/python', '/content/extract_indicwav2vec_v7.py', '/content/iw2v_v7_manifest.jsonl']' returned non-zero exit status 1.

In [21]:
def run_iw2v_cache(items, manifest='/content/iw2v_v7_manifest.jsonl'):
    missing = bad_or_missing(items)

    print(
        f'Indic cache required {len(items)} | '
        f'already good {len(items)-len(missing)} | '
        f'extract/repair {len(missing)}'
    )

    if not missing:
        return

    # IMPORTANT: real newline, not literal "\\n"
    with open(manifest, 'w', encoding='utf-8') as f:
        for g in missing:
            f.write(
                json.dumps({
                    'cid': g['cid'],
                    'path': g['path']
                }) + '\n'
            )

    # sanity check
    with open(manifest, encoding='utf-8') as f:
        lines = [x.strip() for x in f if x.strip()]

    print("manifest records:", len(lines))
    print("first record:", json.loads(lines[0]))

    assert len(lines) == len(missing)

    ensure_iw2v_runtime()

    subprocess.run(
        [
            IW2V_ENV + '/bin/python',
            EXTRACT_SCRIPT,
            manifest
        ],
        check=True
    )

    left = bad_or_missing(items)

    assert not left, \
        f'{len(left)} invalid Indic features remain'

    print("✓ Indic cache complete")

In [22]:
REQUIRED_IW2V = sup_train + gva + sva
run_iw2v_cache(REQUIRED_IW2V)

Indic cache required 10892 | already good 3600 | extract/repair 7292
manifest records: 7292
first record: {'cid': 'g006424', 'path': '/content/aud/g006424.npy'}
✓ Indic cache complete


## 7 · IndicWav2Vec occupancy companion

This branch is deliberately **frame/segment-oriented**: it predicts only event occupancy and is trained with focal BCE + a stronger Dice term. WavLM still owns onset/offset predictions.

In [23]:
def aligned_iw2v_crop(g,T_full,st,T_crop):
    z=np.load(f"{IW2V_FEAT_DIR}/{g['cid']}.npy").astype(np.float32)
    # Map 10-ms mel crop coordinates into the native ~20-ms Indic feature timeline.
    r=len(z)/max(T_full,1)
    a=max(0,int(np.floor(st*r)))
    b=min(len(z),max(a+2,int(np.ceil((st+T_crop)*r))))
    zz=z[a:b]
    # Return exactly one representation vector per 10-ms mel frame.
    t=torch.from_numpy(zz.T).unsqueeze(0)
    t=F.interpolate(t,size=T_crop,mode='linear',align_corners=False)
    return t.squeeze(0).T.contiguous()  # [T_crop,768]

class IndicOccSup(Dataset):
    def __init__(self,items,train=True): self.items=items; self.train=train
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        g=self.items[i]
        y=np.load(g['path']).astype(np.float32)
        x=logmel(y); T_full=x.shape[1]
        frame=np.zeros(T_full,np.float32)
        for on,off,_,_ in g['events_loc']:
            a=max(0,int(round(on/FRAME))); b=min(T_full,int(round(off/FRAME)))
            if b>a: frame[a:b]=1.0
        st=0; T_crop=T_full
        if T_full>MAX_FR:
            is_syn=str(g['cid']).startswith('y')
            if self.train and not is_syn: st=choose_natural_crop(frame,g['events_loc'],T_full)
            else: st=random.randint(0,T_full-MAX_FR) if self.train else 0
            T_crop=MAX_FR
        xx=x[:,st:st+T_crop]; yy=frame[st:st+T_crop]
        xn=(xx-xx.mean())/(xx.std()+1e-5)
        if self.train: xn=spec_augment(xn)
        z=aligned_iw2v_crop(g,T_full,st,T_crop)
        return torch.from_numpy(xn),torch.from_numpy(yy),z

def collate_indic(b):
    T=max(x[0].shape[1] for x in b); B=len(b)
    X=torch.zeros(B,1,N_MELS,T); Y=torch.zeros(B,T); M=torch.zeros(B,T)
    Z=torch.zeros(B,T,IW2V_DIM)
    for i,(x,y,z) in enumerate(b):
        n=x.shape[1]; X[i,0,:,:n]=x; Y[i,:n]=y; M[i,:n]=1; Z[i,:n]=z[:n]
    return X,Y,M,Z

class IndicOccupancyNet(nn.Module):
    def __init__(self,n_mels=N_MELS,ch=(32,64,128,128),rnn=224,proj=160):
        super().__init__(); blk=[]; cin=1
        for c in ch:
            blk += [nn.Conv2d(cin,c,3,padding=1),nn.BatchNorm2d(c),nn.SiLU(),
                    nn.AvgPool2d((2,1)),nn.Dropout2d(.10)]
            cin=c
        self.cnn=nn.Sequential(*blk)
        self.mel_feat=cin*(n_mels//2**len(ch))
        self.indic_proj=nn.Sequential(nn.Linear(IW2V_DIM,proj),nn.LayerNorm(proj),nn.SiLU(),nn.Dropout(.10))
        self.rnn=nn.GRU(self.mel_feat+proj,rnn,2,batch_first=True,bidirectional=True,dropout=.20)
        self.norm=nn.LayerNorm(2*rnn); self.frame_head=nn.Linear(2*rnn,1)
    def forward(self,x,z):
        h=self.cnn(x); B,C,Fq,T=h.shape
        h=h.permute(0,3,1,2).reshape(B,T,C*Fq)
        if z.shape[1]!=T: z=F.interpolate(z.transpose(1,2),size=T,mode='linear',align_corners=False).transpose(1,2)
        z=self.indic_proj(z)
        h,_=self.rnn(torch.cat([h,z],dim=-1)); h=self.norm(h)
        return self.frame_head(h).squeeze(-1)

_tmp=IndicOccupancyNet()
print('Indic companion trainable params:',sum(p.numel() for p in _tmp.parameters())/1e6,'M')
del _tmp

Indic companion trainable params: 2.478433 M


In [24]:
def fit_indic_occ(tag,seed=0):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    model=IndicOccupancyNet().cuda(); path=f'{DUAL_CKPT}/{tag}.pt'
    if os.path.exists(path):
        model.load_state_dict(torch.load(path,map_location='cuda')); model.eval()
        print(tag,': loaded existing checkpoint'); return model
    sampler=WeightedRandomSampler(SUP_WEIGHTS,len(sup_train),replacement=True)
    tr=DataLoader(IndicOccSup(sup_train,True),batch_size=INDIC_BATCH,sampler=sampler,
                  collate_fn=collate_indic,num_workers=0,drop_last=True)
    va_items=gva+sva
    vl=DataLoader(IndicOccSup(va_items,False),batch_size=INDIC_BATCH,shuffle=False,
                  collate_fn=collate_indic,num_workers=0)
    opt=torch.optim.AdamW(model.parameters(),lr=INDIC_LR,weight_decay=1e-2)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=INDIC_LR,total_steps=INDIC_EPOCHS*len(tr),pct_start=.12)
    scaler=torch.amp.GradScaler('cuda'); best=1e9
    for ep in range(INDIC_EPOCHS):
        model.train(); tot=n=0
        for X,Y,M,Z in tr:
            X,Y,M,Z=[q.cuda(non_blocking=True) for q in (X,Y,M,Z)]
            with torch.amp.autocast('cuda'):
                o=model(X,Z)
                loss=masked_focal_bce(o,Y,M)+INDIC_DICE_W*soft_dice_loss(o,Y,M)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            scaler.step(opt); scaler.update(); sch.step(); tot+=loss.item()*len(X); n+=len(X)
        model.eval(); vt=vn=0
        with torch.no_grad():
            for X,Y,M,Z in vl:
                X,Y,M,Z=[q.cuda() for q in (X,Y,M,Z)]
                with torch.amp.autocast('cuda'):
                    o=model(X,Z); v=masked_focal_bce(o,Y,M)+INDIC_DICE_W*soft_dice_loss(o,Y,M)
                vt+=v.item()*len(X); vn+=len(X)
        vv=vt/max(vn,1); flag=''
        if vv<best: best=vv; torch.save(model.state_dict(),path); flag=' *'
        print(f'{tag} ep {ep+1:02d} train={tot/max(n,1):.4f} val={vv:.4f}{flag}')
    model.load_state_dict(torch.load(path,map_location='cuda')); model.eval(); return model

indic_models=[fit_indic_occ(f'indic_occ_s{s}',s) for s in INDIC_SEEDS]
print('Indic occupancy ensemble:',len(indic_models))

indic_occ_s0 ep 01 train=0.6649 val=0.6042 *
indic_occ_s0 ep 02 train=0.6042 val=0.5795 *
indic_occ_s0 ep 03 train=0.5702 val=0.5662 *
indic_occ_s0 ep 04 train=0.5565 val=0.5376 *
indic_occ_s0 ep 05 train=0.5372 val=0.5356 *
indic_occ_s0 ep 06 train=0.5206 val=0.5269 *
indic_occ_s0 ep 07 train=0.5014 val=0.5199 *
indic_occ_s0 ep 08 train=0.4819 val=0.5162 *
indic_occ_s0 ep 09 train=0.4592 val=0.5390
indic_occ_s0 ep 10 train=0.4406 val=0.5304


KeyboardInterrupt: 

In [28]:
indic_models = [
    fit_indic_occ(f'indic_occ_s{s}', s)
    for s in INDIC_SEEDS
]

indic_occ_s0 : loaded existing checkpoint
indic_occ_s1 : loaded existing checkpoint


## 8 · Extract WavLM boundary bundles and Indic occupancy posteriors

In [29]:
def infer_wavlm(model,y):
    x=logmel(y); x=(x-x.mean())/(x.std()+1e-5); T=x.shape[1]
    need=T*HOP; yy=np.pad(y,(0,max(0,need-len(y))))[:need]
    xt=torch.from_numpy(x)[None,None].cuda(); wt=torch.from_numpy(yy.astype(np.float32))[None].cuda()
    with torch.no_grad(),torch.amp.autocast('cuda'): o=model(xt,wt)
    return {k:torch.sigmoid(v)[0].float().cpu().numpy() for k,v in o.items()}

def wavlm_bundle(items):
    out={}
    for j,g in enumerate(items):
        y=np.load(g['path']).astype(np.float32); pp=[infer_wavlm(m,y) for m in models]
        L=min(len(z['frame']) for z in pp)
        out[g['cid']]={k:np.mean([z[k][:L] for z in pp],axis=0) for k in ('frame','onset','offset')}
        if j%100==0: print('WavLM',j,'/',len(items),end='\r')
    print(); return out

@torch.inference_mode()
def infer_indic_occ(model,g,y=None):
    if y is None: y=np.load(g['path']).astype(np.float32)
    x=logmel(y); x=(x-x.mean())/(x.std()+1e-5); T=x.shape[1]
    z=np.load(f"{IW2V_FEAT_DIR}/{g['cid']}.npy").astype(np.float32)
    zt=torch.from_numpy(z.T).unsqueeze(0)
    zt=F.interpolate(zt,size=T,mode='linear',align_corners=False).transpose(1,2)
    X=torch.from_numpy(x)[None,None].cuda(); Z=zt.cuda()
    with torch.amp.autocast('cuda'): o=model(X,Z)
    return torch.sigmoid(o)[0].float().cpu().numpy()

def indic_bundle(items):
    out={}
    for j,g in enumerate(items):
        pp=[infer_indic_occ(m,g) for m in indic_models]
        L=min(map(len,pp)); out[g['cid']]=np.mean([p[:L] for p in pp],axis=0)
        if j%100==0: print('Indic',j,'/',len(items),end='\r')
    print(); return out

POST_N_W=wavlm_bundle(gva); POST_S_W=wavlm_bundle(sva)
POST_N_I=indic_bundle(gva); POST_S_I=indic_bundle(sva)
print('natural bundles:',len(POST_N_W),len(POST_N_I),'| synthetic:',len(POST_S_W),len(POST_S_I))





natural bundles: 1416 1416 | synthetic: 762 762


## 9 · Decoder and complementary frame fusion

Only the **frame occupancy posterior** is fused:

\[
p_{frame}=lpha p_{WavLM} + (1-lpha)p_{IndicW2V}.
\]

WavLM onset/offset posteriors are left untouched. `alpha=1.0` is the exact WavLM-only control.

In [30]:
def hysteresis_regions(p,hi,lo):
    active=False; s=None; regs=[]
    for i,v in enumerate(p):
        if not active and v>=hi:
            active=True; s=i
        elif active and v<lo:
            regs.append([s,i]); active=False; s=None
    if active: regs.append([s,len(p)])
    return regs

def snap_peak(arr,center,radius):
    a=max(0,center-radius); b=min(len(arr),center+radius+1)
    if b<=a: return center
    return a+int(np.argmax(arr[a:b]))

def decode_v6(bundle,dur,hi=0.72,lo=0.34,min_dur=0.08,merge_gap=0.10,snap_ms=180,pad_ms=0):
    p,onp,offp=bundle['frame'],bundle['onset'],bundle['offset']
    regs=hysteresis_regions(p,hi,lo)
    rad=max(1,int(round((snap_ms/1000)/FRAME)))
    snapped=[]
    for a,b in regs:
        aa=snap_peak(onp,a,rad)
        bb=snap_peak(offp,max(a,b-1),rad)+1
        # protect against pathological crossing; fall back to hysteresis boundary
        if bb<=aa: aa,bb=a,b
        snapped.append([aa*FRAME,bb*FRAME])
    merged=[]
    for a,b in snapped:
        if merged and a-merged[-1][1]<=merge_gap: merged[-1][1]=max(merged[-1][1],b)
        else: merged.append([a,b])
    pad=pad_ms/1000
    return [(max(0,a-pad),min(dur,b+pad)) for a,b in merged if b-a>=min_dur]

GRID=list(itertools.product(
    [0.58,0.64,0.70,0.76,0.82],       # hi
    [0.22,0.28,0.34,0.40,0.46],       # lo
    [0.05,0.08,0.12,0.18],            # min duration
    [0.04,0.08,0.12,0.18],            # merge gap
    [100,160,220]                      # snap radius ms
))
GRID=[g for g in GRID if g[1]<g[0]]

def tune_decoder(posts,items):
    refs={g['cid']:[(s,e) for s,e,_,_ in g['events']] for g in items}
    durs={g['cid']:g['dur'] for g in items}
    rows=[]
    for hi,lo,mind,mg,snap in GRID:
        pred={c:decode_v6(posts[c],durs[c],hi,lo,mind,mg,snap) for c in posts}
        s=score(pred,refs,durs,verbose=False)
        rows.append((s['combined'],s['f1'],s['dice'],hi,lo,mind,mg,snap,s['precision'],s['recall']))
    rows.sort(reverse=True)
    return rows



In [31]:
def half_split(items,seed):
    idx=np.arange(len(items)); a,b=train_test_split(idx,test_size=.5,random_state=seed)
    return [items[i] for i in a],[items[i] for i in b]

ncal,neval=half_split(gva,3101); scal,seval=half_split(sva,3102)

def fused_posts(wposts,iposts,alpha,ids=None):
    keys=list(wposts) if ids is None else [c for c in ids if c in wposts]
    out={}
    for c in keys:
        wb=wposts[c]; ip=iposts[c]; L=min(len(wb['frame']),len(ip))
        out[c]={
            'frame':np.clip(alpha*wb['frame'][:L]+(1-alpha)*ip[:L],0,1),
            'onset':wb['onset'][:L],
            'offset':wb['offset'][:L],
        }
    return out

def tune_alpha_and_decoder(wposts,iposts,items,alphas):
    ids=[g['cid'] for g in items]; rows=[]
    for a in alphas:
        fp=fused_posts(wposts,iposts,a,ids)
        rr=tune_decoder(fp,items)
        best=rr[0]
        rows.append((best[0],best[1],best[2],a,best[3],best[4],best[5],best[6],best[7],best[8],best[9]))
        print(f'alpha={a:.2f} combined={best[0]:.4f} f1={best[1]:.4f} dice={best[2]:.4f}')
    rows.sort(reverse=True); return rows

N_ROWS_V7=tune_alpha_and_decoder(POST_N_W,POST_N_I,ncal,ALPHAS_NAT)
S_ROWS_V7=tune_alpha_and_decoder(POST_S_W,POST_S_I,scal,ALPHAS_SYN)

# Exact WavLM-only controls, calibrated on the same halves.
N_ROWS_CTRL=tune_alpha_and_decoder(POST_N_W,POST_N_I,ncal,[1.0])
S_ROWS_CTRL=tune_alpha_and_decoder(POST_S_W,POST_S_I,scal,[1.0])

def fusion_row_cfg(r):
    return r[3],dict(hi=r[4],lo=r[5],min_dur=r[6],merge_gap=r[7],snap_ms=r[8])

A_NAT,P_NAT=fusion_row_cfg(N_ROWS_V7[0]); A_SYN,P_SYN=fusion_row_cfg(S_ROWS_V7[0])
_,P_NAT_CTRL=fusion_row_cfg(N_ROWS_CTRL[0]); _,P_SYN_CTRL=fusion_row_cfg(S_ROWS_CTRL[0])
print('\nV7 NAT alpha/cfg:',A_NAT,P_NAT)
print('V7 SYN alpha/cfg:',A_SYN,P_SYN)
print('Control NAT:',P_NAT_CTRL)
print('Control SYN:',P_SYN_CTRL)

alpha=1.00 combined=0.8725 f1=0.3072 dice=0.5654
alpha=0.95 combined=0.8832 f1=0.3073 dice=0.5759
alpha=0.90 combined=0.8856 f1=0.3085 dice=0.5771
alpha=0.85 combined=0.8896 f1=0.3122 dice=0.5775
alpha=0.80 combined=0.8991 f1=0.3158 dice=0.5832
alpha=0.75 combined=0.9051 f1=0.3193 dice=0.5858
alpha=0.70 combined=0.9102 f1=0.3203 dice=0.5899
alpha=0.60 combined=0.9119 f1=0.3185 dice=0.5933
alpha=1.00 combined=1.0539 f1=0.3719 dice=0.6820
alpha=0.90 combined=1.0715 f1=0.3955 dice=0.6760
alpha=0.80 combined=1.0874 f1=0.3934 dice=0.6940
alpha=0.70 combined=1.1142 f1=0.4089 dice=0.7053
alpha=0.60 combined=1.1182 f1=0.4130 dice=0.7051
alpha=0.50 combined=1.1210 f1=0.4133 dice=0.7077
alpha=0.40 combined=1.1258 f1=0.4133 dice=0.7125
alpha=0.30 combined=1.1188 f1=0.4120 dice=0.7068
alpha=1.00 combined=0.8725 f1=0.3072 dice=0.5654
alpha=1.00 combined=1.0539 f1=0.3719 dice=0.6820

V7 NAT alpha/cfg: 0.6 {'hi': 0.58, 'lo': 0.46, 'min_dur': 0.12, 'merge_gap': 0.18, 'snap_ms': 100}
V7 SYN alpha/cfg: 

In [34]:
print([k for k in globals() if "post" in k.lower()])

['POST_N_W', 'POST_S_W', 'POST_N_I', 'POST_S_I', 'fused_posts']


In [33]:
# Load posterior dictionaries
POST_N = np.load(NCAL_POST_PATH, allow_pickle=True).item()
POST_S = np.load(SCAL_POST_PATH, allow_pickle=True).item()

print("POST_N:", len(POST_N))
print("POST_S:", len(POST_S))

NameError: name 'NCAL_POST_PATH' is not defined

## 10 · Natural/synthetic gate

The gate is deliberately trained from the **WavLM-only bundle**, so domain classification is independent of the new fusion weight. It selects the natural or synthetic fusion/decoder policy; it never deletes events.

In [32]:
def gate_features(y,b):
    p,on,off=b['frame'],b['onset'],b['offset']
    feats=[]
    for a in (p,on,off):
        feats += [float(a.mean()),float(a.std()),float(a.max()),
                  float(np.quantile(a,.75)),float(np.quantile(a,.90)),float(np.quantile(a,.98))]
    feats += [float((p>.5).mean()),float((p>.7).mean()),float((p>.85).mean())]
    # light acoustic features; synthetic overlays often have sharper local distribution changes
    rms=float(np.sqrt(np.mean(y*y)+1e-12))
    zcr=float(librosa.feature.zero_crossing_rate(y,frame_length=512,hop_length=256).mean())
    flat=float(librosa.feature.spectral_flatness(y=y,n_fft=512,hop_length=256).mean())
    cent=float(librosa.feature.spectral_centroid(y=y,sr=SR,n_fft=512,hop_length=256).mean()/SR)
    feats += [rms,zcr,flat,cent]
    return np.asarray(feats,np.float32)

X=[];Y=[];C=[]
for g in ncal:
    y=np.load(g['path']).astype(np.float32); X.append(gate_features(y,POST_N[g['cid']])); Y.append(0); C.append(g['cid'])
for g in scal:
    y=np.load(g['path']).astype(np.float32); X.append(gate_features(y,POST_S[g['cid']])); Y.append(1); C.append(g['cid'])
X=np.stack(X); Y=np.asarray(Y)
GATE=RandomForestClassifier(n_estimators=500,min_samples_leaf=4,max_features='sqrt',
                            class_weight='balanced',random_state=SEED,n_jobs=-1)
GATE.fit(X,Y)

# choose threshold for >=95% synthetic precision when possible
prob=GATE.predict_proba(X)[:,1]
pr,re,th=precision_recall_curve(Y,prob)
choices=[]
for i,t in enumerate(th):
    if pr[i]>=0.95: choices.append((re[i],float(t),pr[i]))
GATE_THR=max(choices,key=lambda z:z[0])[1] if choices else 0.80
print('Gate threshold:',GATE_THR)

NameError: name 'POST_N' is not defined

In [35]:
def gate_features(y,b):
    p,on,off=b['frame'],b['onset'],b['offset']
    feats=[]
    for a in (p,on,off):
        feats += [float(a.mean()),float(a.std()),float(a.max()),
                  float(np.quantile(a,.75)),float(np.quantile(a,.90)),float(np.quantile(a,.98))]
    feats += [float((p>.5).mean()),float((p>.7).mean()),float((p>.85).mean())]
    # light acoustic features; synthetic overlays often have sharper local distribution changes
    rms=float(np.sqrt(np.mean(y*y)+1e-12))
    zcr=float(librosa.feature.zero_crossing_rate(y,frame_length=512,hop_length=256).mean())
    flat=float(librosa.feature.spectral_flatness(y=y,n_fft=512,hop_length=256).mean())
    cent=float(librosa.feature.spectral_centroid(y=y,sr=SR,n_fft=512,hop_length=256).mean()/SR)
    feats += [rms,zcr,flat,cent]
    return np.asarray(feats,np.float32)

X=[];Y=[];C=[]
for g in ncal:
    y=np.load(g['path']).astype(np.float32); X.append(gate_features(y,POST_N_W[g['cid']])); Y.append(0); C.append(g['cid'])
for g in scal:
    y=np.load(g['path']).astype(np.float32); X.append(gate_features(y,POST_S_W[g['cid']])); Y.append(1); C.append(g['cid'])
X=np.stack(X); Y=np.asarray(Y)
GATE=RandomForestClassifier(n_estimators=500,min_samples_leaf=4,max_features='sqrt',
                            class_weight='balanced',random_state=SEED,n_jobs=-1)
GATE.fit(X,Y)

# choose threshold for >=95% synthetic precision when possible
prob=GATE.predict_proba(X)[:,1]
pr,re,th=precision_recall_curve(Y,prob)
choices=[]
for i,t in enumerate(th):
    if pr[i]>=0.95: choices.append((re[i],float(t),pr[i]))
GATE_THR=max(choices,key=lambda z:z[0])[1] if choices else 0.80
print('Gate threshold:',GATE_THR)

Gate threshold: 0.5106806901556886


## 11 · Honest held-out V7 vs WavLM-only control

In [36]:
def adaptive_v7(g,wposts,iposts):
    wb=wposts[g['cid']]; y=np.load(g['path']).astype(np.float32)
    ps=float(GATE.predict_proba(gate_features(y,wb)[None])[0,1])
    syn_mode=ps>=GATE_THR
    a=A_SYN if syn_mode else A_NAT; cfg=P_SYN if syn_mode else P_NAT
    b=fused_posts({g['cid']:wb},{g['cid']:iposts[g['cid']]},a)[g['cid']]
    return decode_v6(b,g['dur'],**cfg),ps

def adaptive_control(g,wposts):
    wb=wposts[g['cid']]; y=np.load(g['path']).astype(np.float32)
    ps=float(GATE.predict_proba(gate_features(y,wb)[None])[0,1])
    cfg=P_SYN_CTRL if ps>=GATE_THR else P_NAT_CTRL
    return decode_v6(wb,g['dur'],**cfg),ps

def eval_policy(items,wposts,iposts,mode='v7'):
    pred={}; pp={}
    for g in items:
        if mode=='v7': ev,p=adaptive_v7(g,wposts,iposts)
        else: ev,p=adaptive_control(g,wposts)
        pred[g['cid']]=ev; pp[g['cid']]=p
    ref={g['cid']:[(s,e) for s,e,_,_ in g['events']] for g in items}; dur={g['cid']:g['dur'] for g in items}
    return score(pred,ref,dur,verbose=False),pred,pp,ref,dur

rows=[]
for mode,label in [('control','WavLM-only control'),('v7','WavLM + IndicW2V fusion')]:
    n,pn,gn,rn,dn=eval_policy(neval,POST_N_W,POST_N_I,mode)
    s,ps,gs,rs,ds=eval_policy(seval,POST_S_W,POST_S_I,mode)
    pall={**pn,**ps}; rall={**rn,**rs}; dall={**dn,**ds}
    a=score(pall,rall,dall,verbose=False)
    rows.append(dict(system=label,nat_f1=n['f1'],nat_dice=n['dice'],nat_comb=n['combined'],
                     syn_f1=s['f1'],syn_dice=s['dice'],syn_comb=s['combined'],
                     all_f1=a['f1'],all_dice=a['dice'],combined=a['combined']))
RES_V7=pd.DataFrame(rows)
display(RES_V7.round(4))
base=RES_V7.iloc[0]; v7=RES_V7.iloc[1]
print('Δ V7-control:',{c:round(float(v7[c]-base[c]),4) for c in ['nat_f1','nat_dice','syn_f1','syn_dice','combined']})
print('Chosen alphas: natural',A_NAT,'synthetic',A_SYN)
if v7['combined']<=base['combined']:
    print('STOP: fusion did not beat the same-split WavLM-only control. Do not create a leaderboard submission.')
else:
    print('PASS: fusion beats the same-split control. Proceed to test inference.')

,system,nat_f1,nat_dice,nat_comb,syn_f1,syn_dice,syn_comb,all_f1,all_dice,combined
0,WavLM-only control,0.3157,0.5911,0.9068,0.3820,0.6536,1.0356,0.3385,0.6129,0.9515
1,WavLM + IndicW2V fusion,0.3113,0.6096,0.9209,0.4003,0.6807,1.0810,0.3422,0.6345,0.9766


Δ V7-control: {'nat_f1': -0.0044, 'nat_dice': 0.0185, 'syn_f1': 0.0184, 'syn_dice': 0.0271, 'combined': 0.0252}
Chosen alphas: natural 0.6 synthetic 0.4
PASS: fusion beats the same-split control. Proceed to test inference.


## 12 · Diagnostics before submission

A useful V7 should normally improve **Dice without sacrificing too much event F1**. Also inspect whether the tuned natural alpha stays close to WavLM (`alpha` high) while synthetic alpha allows more Indic contribution.

In [37]:
print(RES_V7.round(4).to_string(index=False))
print('\nTop natural fusion settings:')
display(pd.DataFrame(N_ROWS_V7[:8],columns=['combined','f1','dice','alpha','hi','lo','min_dur','merge_gap','snap_ms','precision','recall']).round(4))
print('\nTop synthetic fusion settings:')
display(pd.DataFrame(S_ROWS_V7[:8],columns=['combined','f1','dice','alpha','hi','lo','min_dur','merge_gap','snap_ms','precision','recall']).round(4))
print('\nReference supplied earlier: your strong system ≈ 1.3617 combined; visible leaderboard leader ≈ 1.52.')

                 system  nat_f1  nat_dice  nat_comb  syn_f1  syn_dice  syn_comb  all_f1  all_dice  combined
     WavLM-only control  0.3157    0.5911    0.9068  0.3820    0.6536    1.0356  0.3385    0.6129    0.9515
WavLM + IndicW2V fusion  0.3113    0.6096    0.9209  0.4003    0.6807    1.0810  0.3422    0.6345    0.9766

Top natural fusion settings:


,combined,f1,dice,alpha,hi,lo,min_dur,merge_gap,snap_ms,precision,recall
0,0.9119,0.3185,0.5933,0.60,0.58,0.46,0.12,0.18,100,0.3576,0.2872
1,0.9102,0.3203,0.5899,0.70,0.58,0.46,0.12,0.18,100,0.3584,0.2895
2,0.9051,0.3193,0.5858,0.75,0.58,0.46,0.12,0.18,100,0.3571,0.2887
3,0.8991,0.3158,0.5832,0.80,0.58,0.46,0.12,0.18,100,0.3532,0.2856
4,0.8896,0.3122,0.5775,0.85,0.58,0.46,0.12,0.18,160,0.3477,0.2833
5,0.8856,0.3085,0.5771,0.90,0.58,0.46,0.12,0.18,100,0.3432,0.2802
6,0.8832,0.3073,0.5759,0.95,0.58,0.46,0.12,0.18,100,0.3403,0.2802
7,0.8725,0.3072,0.5654,1.00,0.58,0.46,0.12,0.18,220,0.3399,0.2802



Top synthetic fusion settings:


,combined,f1,dice,alpha,hi,lo,min_dur,merge_gap,snap_ms,precision,recall
0,1.1258,0.4133,0.7125,0.4,0.58,0.46,0.18,0.04,160,0.4407,0.3891
1,1.1210,0.4133,0.7077,0.5,0.58,0.46,0.18,0.04,100,0.4429,0.3875
2,1.1188,0.4120,0.7068,0.3,0.58,0.46,0.18,0.04,160,0.4397,0.3875
3,1.1182,0.4130,0.7051,0.6,0.58,0.46,0.18,0.18,160,0.4442,0.3859
4,1.1142,0.4089,0.7053,0.7,0.58,0.46,0.12,0.18,160,0.4349,0.3859
5,1.0874,0.3934,0.6940,0.8,0.58,0.46,0.12,0.18,100,0.4195,0.3703
6,1.0715,0.3955,0.6760,0.9,0.64,0.46,0.12,0.18,160,0.4444,0.3562
7,1.0539,0.3719,0.6820,1.0,0.58,0.46,0.12,0.04,160,0.3947,0.3516



Reference supplied earlier: your strong system ≈ 1.3617 combined; visible leaderboard leader ≈ 1.52.


## 13 · Cache IndicWav2Vec test features

Run this **only after the V7 holdout beats the control**. Existing test features are reused; only missing/invalid 768-D files are extracted.

In [38]:
TEST_LOCAL_CANDIDATES=['/content/indoml_test_local','/content/test_wavs','/content/synthetic_hunter/test_wavs']
test_local=next((p for p in TEST_LOCAL_CANDIDATES if os.path.isdir(p) and glob.glob(p+'/**/*.wav',recursive=True)),None)
if test_local:
    wavs=sorted(glob.glob(test_local+'/**/*.wav',recursive=True)); print('Using local test WAVs:',test_local)
else:
    wavs=sorted(glob.glob(f'{TEST_ROOT}/**/*.wav',recursive=True)); print('Using Drive test WAVs:',TEST_ROOT)
print('test clips:',len(wavs))

test_items=[]
for p in wavs:
    c=os.path.splitext(os.path.basename(p))[0]
    test_items.append({'cid':c,'path':p})
run_iw2v_cache(test_items,'/content/iw2v_v7_test_manifest.jsonl')
print('✓ test Indic cache ready')

Using Drive test WAVs: /content/drive/MyDrive/PHD/Datathon/
test clips: 5517
Indic cache required 5517 | already good 0 | extract/repair 5517
manifest records: 5517
first record: {'cid': 'IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213769_001_GENERIC_0098_17486_21246', 'path': '/content/drive/MyDrive/PHD/Datathon/IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213769_001_GENERIC_0098_17486_21246.wav'}
✓ Indic cache complete
✓ test Indic cache ready


## 14 · V7 test inference and submission

In [39]:
def infer_indic_test(model,c,y):
    x=logmel(y); x=(x-x.mean())/(x.std()+1e-5); T=x.shape[1]
    z=np.load(f'{IW2V_FEAT_DIR}/{c}.npy').astype(np.float32)
    zt=torch.from_numpy(z.T).unsqueeze(0)
    zt=F.interpolate(zt,size=T,mode='linear',align_corners=False).transpose(1,2)
    X=torch.from_numpy(x)[None,None].cuda(); Z=zt.cuda()
    with torch.no_grad(),torch.amp.autocast('cuda'): o=model(X,Z)
    return torch.sigmoid(o)[0].float().cpu().numpy()

recs=[]; diag=[]; t0=time.time()
for i,p in enumerate(wavs):
    c=os.path.splitext(os.path.basename(p))[0]
    y,_=librosa.load(p,sr=SR,mono=True); y=y.astype(np.float32); dur=len(y)/SR
    # WavLM ensemble bundle
    wb0=[infer_wavlm(m,y) for m in models]; Lw=min(len(z['frame']) for z in wb0)
    wb={k:np.mean([z[k][:Lw] for z in wb0],axis=0) for k in ('frame','onset','offset')}
    # Indic occupancy ensemble
    ip0=[infer_indic_test(m,c,y) for m in indic_models]; Li=min(map(len,ip0)); ip=np.mean([z[:Li] for z in ip0],axis=0)
    ps=float(GATE.predict_proba(gate_features(y,wb)[None])[0,1]); syn_mode=ps>=GATE_THR
    a=A_SYN if syn_mode else A_NAT; cfg=P_SYN if syn_mode else P_NAT
    b=fused_posts({c:wb},{c:ip},a)[c]
    ev=decode_v6(b,dur,**cfg)
    recs.append({'clip_id':c,'events':[{'onset':round(float(x),3),'offset':round(float(y_),3)} for x,y_ in ev]})
    diag.append({'clip_id':c,'p_syn':ps,'decoder':'syn' if syn_mode else 'nat','alpha':a,'n_events':len(ev)})
    if i%100==0: print(f'{i}/{len(wavs)} {(time.time()-t0)/60:.1f} min',end='\r')

OUT_JL=f'{WORK}/submission_stage7_wavlm_indicw2v_fusion.jsonl'
with open(OUT_JL,'w',encoding='utf-8') as f:
    for r in recs: f.write(json.dumps(r,ensure_ascii=False)+'\n')
with open(f'{WORK}/submission_stage7_wavlm_indicw2v_fusion_diag.jsonl','w',encoding='utf-8') as f:
    for r in diag: f.write(json.dumps(r,ensure_ascii=False)+'\n')
print('\nSaved:',OUT_JL)
print('events:',sum(len(r['events']) for r in recs),'| empty:',sum(not r['events'] for r in recs))
print('decoder usage:',pd.Series([r['decoder'] for r in diag]).value_counts().to_dict())
print('alpha usage:',pd.Series([r['alpha'] for r in diag]).value_counts().to_dict())


Saved: /content/drive/MyDrive/Datathon/work/submission_stage7_wavlm_indicw2v_fusion.jsonl
events: 12106 | empty: 174
decoder usage: {'nat': 3294, 'syn': 2223}
alpha usage: {0.6: 3294, 0.4: 2223}


In [40]:
import zipfile, os

zip_path = f"{WORK}/submission_stage7_wavlm_indicw2v_fusion.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(
        OUT_JL,
        arcname=os.path.basename(OUT_JL)
    )

print("Saved:", zip_path)

Saved: /content/drive/MyDrive/Datathon/work/submission_stage7_wavlm_indicw2v_fusion.zip


In [41]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

y_true = []
y_prob = []

for g in nhold:
    y = np.load(g['path']).astype(np.float32)
    p = GATE.predict_proba(
        gate_features(y, POST_N_H[g['cid']])[None]
    )[0,1]
    y_true.append(0)
    y_prob.append(p)

for g in shold:
    y = np.load(g['path']).astype(np.float32)
    p = GATE.predict_proba(
        gate_features(y, POST_S_H[g['cid']])[None]
    )[0,1]
    y_true.append(1)
    y_prob.append(p)

y_pred = (np.array(y_prob) >= GATE_THR).astype(int)

print("threshold:", GATE_THR)
print("accuracy :", accuracy_score(y_true, y_pred))
print("precision:", precision_score(y_true, y_pred))
print("recall   :", recall_score(y_true, y_pred))
print("F1       :", f1_score(y_true, y_pred))
print("CM:\n", confusion_matrix(y_true, y_pred))

NameError: name 'nhold' is not defined

## Decision rule

Submit V7 only if all of the following hold:

1. **Combined** improves over the same-split `alpha=1.0` WavLM control.
2. Natural F1 does not collapse; ideally Dice is the main gain.
3. The selected `alpha` is not at a pathological extreme caused by calibration leakage.
4. Test event counts / empty-clip rate remain plausible relative to your strongest submission.

If fusion is positive, the next experiment is a learned dual-representation model: **Mel + WavLM + IndicWav2Vec for frame occupancy, but Mel + WavLM only for onset/offset heads**. Late fusion is intentionally tested first because it preserves the already-strong V6 boundary model and provides a clean ablation.